# AstroLogic - RL Training on Kaggle

Trains **30 hyperparameter experiments** across 3 algorithms:

| Algorithm | Runs | Action space | Budget |
|-----------|------|--------------|--------|
| DQN | 10 | `Discrete(1080)` wrapper | 500 K timesteps |
| PPO | 10 | `MultiDiscrete` native | 500 K timesteps |
| REINFORCE | 10 | `MultiDiscrete` native | 5 000 episodes |

**Environment**: `AstroExploration-v0` - spacecraft navigates the solar system to detect biosignatures on Mars, Europa, and Enceladus.

Set `FAST_MODE = True` for a quick smoke-test (50 K steps / 500 episodes).

In [2]:
import os, sys

FAST_MODE = True         # True → 50 K timesteps / 500 episodes for quick test
TOTAL_TIMESTEPS   = 50_000 if FAST_MODE else 500_000
REINFORCE_EPISODES = 500   if FAST_MODE else 5_000
SEED = 42

WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

for d in ["astro_env", "agents", "training", "evaluation",
          "results/logs", "results/models", "results/plots", "results/diagrams"]:
    os.makedirs(d, exist_ok=True)

print(f"FAST_MODE={FAST_MODE}  TOTAL_TIMESTEPS={TOTAL_TIMESTEPS}  REINFORCE_EPISODES={REINFORCE_EPISODES}")


In [3]:
%%writefile astro_env/__init__.py
"""AstroExploration Gymnasium Environment Package."""

from gymnasium.envs.registration import register

register(
    id="AstroExploration-v0",
    entry_point="astro_env.astro_exploration_env:AstroExplorationEnv",
    max_episode_steps=100_000,
)


In [4]:
%%writefile astro_env/celestial_bodies.py
"""Celestial body data for the AstroExploration environment."""

import numpy as np

CELESTIAL_BODIES = {
    "Sun": {"mass": 1.0, "radius": 0.00465, "orbit_radius": 0.0, "orbit_period": 0.0,
            "initial_angle": 0.0, "color": (255, 223, 0), "parent": None,
            "biosignatures": [], "detection_zone_radius": 0.0},
    "Earth": {"mass": 3.003e-6, "radius": 0.0000426, "orbit_radius": 1.0,
             "orbit_period": 365.25, "initial_angle": 0.0, "color": (70, 130, 200),
             "parent": "Sun", "biosignatures": [], "detection_zone_radius": 0.0},
    "Mars": {"mass": 3.213e-7, "radius": 0.0000227, "orbit_radius": 1.524,
            "orbit_period": 687.0, "initial_angle": np.pi / 4, "color": (193, 68, 14),
            "parent": "Sun", "biosignatures": ["ice", "organic_compounds"],
            "detection_zone_radius": 0.05},
    "Jupiter": {"mass": 9.543e-4, "radius": 0.000467, "orbit_radius": 5.203,
               "orbit_period": 4332.59, "initial_angle": np.pi / 3, "color": (201, 176, 131),
               "parent": "Sun", "biosignatures": [], "detection_zone_radius": 0.0},
    "Europa": {"mass": 2.528e-8, "radius": 0.0000104, "orbit_radius": 0.0045,
              "orbit_period": 3.55, "initial_angle": 0.0, "color": (200, 220, 240),
              "parent": "Jupiter",
              "biosignatures": ["liquid_water", "organic_compounds"],
              "detection_zone_radius": 0.03},
    "Saturn": {"mass": 2.857e-4, "radius": 0.000389, "orbit_radius": 9.537,
              "orbit_period": 10759.22, "initial_angle": 2 * np.pi / 3, "color": (210, 180, 100),
              "parent": "Sun", "biosignatures": [], "detection_zone_radius": 0.0},
    "Enceladus": {"mass": 1.08e-10, "radius": 0.00000168, "orbit_radius": 0.00159,
                 "orbit_period": 1.37, "initial_angle": np.pi / 2, "color": (180, 200, 220),
                 "parent": "Saturn",
                 "biosignatures": ["liquid_water", "ice", "signs_of_intelligence"],
                 "detection_zone_radius": 0.02},
}

BIOSIGNATURE_REWARDS = {
    "liquid_water": 500.0, "ice": 300.0,
    "organic_compounds": 750.0, "signs_of_intelligence": 5000.0,
}

TARGET_BODIES = ["Mars", "Europa", "Enceladus"]

INSTRUMENTS = {
    0: {"name": "None", "detects": []},
    1: {"name": "Spectrometer", "detects": ["liquid_water", "organic_compounds"]},
    2: {"name": "ThermalImager", "detects": ["ice", "liquid_water"]},
    3: {"name": "Drill", "detects": ["organic_compounds", "ice", "signs_of_intelligence"]},
}


In [5]:
%%writefile astro_env/physics.py
"""Physics helpers for orbital mechanics simulation."""

import numpy as np

G_NORMALIZED = 4.0 * np.pi**2 / 365.25**2


def gravitational_acceleration(pos, body_pos, body_mass):
    """Compute gravitational acceleration on spacecraft from a single body."""
    direction = body_pos - pos
    distance = np.linalg.norm(direction)
    if distance < 1e-8:
        return np.zeros(3)
    return G_NORMALIZED * body_mass / (distance**2) * (direction / distance)


def orientation_to_direction(pitch, yaw, roll):
    """Convert Euler angles (radians) to a unit forward direction vector."""
    cos_yaw, sin_yaw = np.cos(yaw), np.sin(yaw)
    cos_pitch, sin_pitch = np.cos(pitch), np.sin(pitch)
    forward = np.array([cos_pitch * cos_yaw, cos_pitch * sin_yaw, sin_pitch])
    return forward / (np.linalg.norm(forward) + 1e-8)


def compute_orbital_position(orbit_radius, orbit_period, time, initial_angle, parent_pos=None):
    """Compute position on a circular orbit at a given time."""
    if orbit_period <= 0 or orbit_radius <= 0:
        return parent_pos.copy() if parent_pos is not None else np.zeros(3)
    angle = initial_angle + 2.0 * np.pi * time / orbit_period
    pos = np.array([orbit_radius * np.cos(angle), orbit_radius * np.sin(angle), 0.0])
    if parent_pos is not None:
        pos += parent_pos
    return pos


def normalize_distance(distance, max_distance=50.0):
    """Normalize distance to [0, 1] range where closer = higher value."""
    return max(0.0, 1.0 - distance / max_distance)


def compute_heading(from_pos, to_pos):
    """Compute unit heading vector from one position to another."""
    direction = to_pos - from_pos
    dist = np.linalg.norm(direction)
    if dist < 1e-8:
        return np.zeros(3)
    return direction / dist


In [6]:
%%writefile astro_env/rewards.py
"""Reward calculator for the AstroExploration environment."""

import numpy as np
from astro_env.celestial_bodies import BIOSIGNATURE_REWARDS


class RewardCalculator:
    """Computes rewards based on environment state transitions."""

    def __init__(
        self,
        step_fuel_penalty=0.005,
        step_time_penalty=0.0005,
        collision_penalty=-600.0,
        orbital_insertion_bonus=100.0,
        transmission_bonus=200.0,
        proximity_scale=0.3,
        direction_bonus_scale=2.0,
        moving_away_penalty=0.25,
        compatible_instrument_bonus=3.0,
        in_zone_hold_bonus=0.5,
        instrument_switch_penalty_far=0.2,
        success_completion_bonus=500.0,
    ):
        self.step_fuel_penalty = step_fuel_penalty
        self.step_time_penalty = step_time_penalty
        self.collision_penalty = collision_penalty
        self.orbital_insertion_bonus = orbital_insertion_bonus
        self.transmission_bonus = transmission_bonus
        self.proximity_scale = proximity_scale
        self.direction_bonus_scale = direction_bonus_scale
        self.moving_away_penalty = moving_away_penalty
        self.compatible_instrument_bonus = compatible_instrument_bonus
        self.in_zone_hold_bonus = in_zone_hold_bonus
        self.instrument_switch_penalty_far = instrument_switch_penalty_far
        self.success_completion_bonus = success_completion_bonus

    def compute(self, state):
        """Compute total reward and breakdown info dict."""
        info = {}
        total = 0.0

        fuel_penalty = -(self.step_fuel_penalty * state.get("fuel_used", 0.0))
        time_penalty = -self.step_time_penalty
        info["reward_step_fuel"] = fuel_penalty
        info["reward_step_time"] = time_penalty
        total += fuel_penalty + time_penalty

        # Dense navigation shaping.
        prev_dist = state.get("prev_min_target_distance", None)
        curr_dist = state.get("min_target_distance", None)
        if prev_dist is not None and curr_dist is not None:
            delta = prev_dist - curr_dist
            if delta > 0:
                direction_bonus = self.direction_bonus_scale * float(delta)
                info["reward_direction"] = direction_bonus
                total += direction_bonus
            else:
                away_penalty = self.moving_away_penalty * float(-delta)
                info["reward_moving_away"] = -away_penalty
                total -= away_penalty

        # Instrument behavior shaping.
        if state.get("in_detection_zone", False) and state.get("instrument_compatible", False):
            info["reward_compatible_instrument"] = self.compatible_instrument_bonus
            total += self.compatible_instrument_bonus

            hold_steps = state.get("instrument_hold_steps", 0)
            if hold_steps >= 2:
                hold_bonus = self.in_zone_hold_bonus * min(hold_steps, 10)
                info["reward_in_zone_hold"] = hold_bonus
                total += hold_bonus

        if state.get("switched_instrument_far", False):
            info["reward_switch_far"] = -self.instrument_switch_penalty_far
            total -= self.instrument_switch_penalty_far

        for biosig in state.get("new_biosignatures", []):
            reward = BIOSIGNATURE_REWARDS.get(biosig, 0.0)
            info[f"reward_detect_{biosig}"] = reward
            total += reward

        for biosig in state.get("new_transmissions", []):
            info[f"reward_transmit_{biosig}"] = self.transmission_bonus
            total += self.transmission_bonus

        if state.get("orbital_insertion", False):
            info["reward_orbital_insertion"] = self.orbital_insertion_bonus
            total += self.orbital_insertion_bonus

        if state.get("collision", False) or state.get("out_of_bounds", False):
            info["reward_collision"] = self.collision_penalty
            total += self.collision_penalty

        min_dist = state.get("min_target_distance", 50.0)
        if min_dist < 8.0:
            proximity_reward = self.proximity_scale * (1.0 / (min_dist + 0.1) - 1.0 / 8.1)
            proximity_reward = max(0.0, proximity_reward)
            info["reward_proximity"] = proximity_reward
            total += proximity_reward

        if state.get("success", False):
            info["reward_success_completion"] = self.success_completion_bonus
            total += self.success_completion_bonus

        info["reward_total"] = total
        return total, info


In [7]:
%%writefile astro_env/wrappers.py
"""Action space wrappers for compatibility with different RL algorithms."""

import numpy as np
import gymnasium as gym
from gymnasium import spaces


class FlattenMultiDiscreteToDiscrete(gym.ActionWrapper):
    """Converts a MultiDiscrete action space to a single Discrete space.

    Required for DQN which only supports Discrete action spaces.
    Uses mixed-radix encoding to map between flat integer and multi-action array.

    For MultiDiscrete([5, 3, 3, 3, 4, 2]):
        Total combinations = 5 * 3 * 3 * 3 * 4 * 2 = 1080
        Flat action 0 -> [0, 0, 0, 0, 0, 0]
        Flat action 1079 -> [4, 2, 2, 2, 3, 1]
    """

    def __init__(self, env):
        super().__init__(env)
        assert isinstance(env.action_space, spaces.MultiDiscrete), (
            "FlattenMultiDiscreteToDiscrete requires a MultiDiscrete action space"
        )
        self.nvec = env.action_space.nvec
        self.total_actions = int(np.prod(self.nvec))
        self.action_space = spaces.Discrete(self.total_actions)

    def action(self, flat_action: int) -> np.ndarray:
        """Decode a flat integer action to a multi-action array."""
        multi_action = np.zeros(len(self.nvec), dtype=np.int64)
        remaining = flat_action
        for i in range(len(self.nvec) - 1, -1, -1):
            multi_action[i] = remaining % self.nvec[i]
            remaining //= self.nvec[i]
        return multi_action

    def reverse_action(self, multi_action: np.ndarray) -> int:
        """Encode a multi-action array to a flat integer."""
        flat = 0
        multiplier = 1
        for i in range(len(self.nvec) - 1, -1, -1):
            flat += int(multi_action[i]) * multiplier
            multiplier *= self.nvec[i]
        return flat


class ReducedDiscreteWrapper(gym.ActionWrapper):
    """Reduces MultiDiscrete([5,3,3,3,4,2]) -> Discrete(360) for DQN.

    Fixes roll to neutral (index 1 = 0 degrees), dropping the roll dimension.
    Resulting nvec: [5, 3, 3, 4, 2] -> 5*3*3*4*2 = 360 actions (vs 1080).
    This gives DQN 3x more samples per action at the same training budget,
    which significantly speeds up Q-value convergence.

    Action layout after reduction:
        [thrust(5), pitch(3), yaw(3), instrument(4), comm(2)]
    Roll is always 1 (neutral, 0 degrees).
    """

    # Index of roll in the original MultiDiscrete: [thrust, pitch, yaw, roll, instrument, comm]
    _ROLL_IDX = 3
    _ROLL_NEUTRAL = 1  # index 1 in ROTATION_DELTAS=[-5, 0, 5] -> 0 degrees

    def __init__(self, env):
        super().__init__(env)
        assert isinstance(env.action_space, spaces.MultiDiscrete), (
            "ReducedDiscreteWrapper requires a MultiDiscrete action space"
        )
        full_nvec = env.action_space.nvec  # [5, 3, 3, 3, 4, 2]
        # Drop roll dimension
        self.reduced_nvec = np.array(
            [n for i, n in enumerate(full_nvec) if i != self._ROLL_IDX], dtype=np.int64
        )
        self.total_actions = int(np.prod(self.reduced_nvec))
        self.action_space = spaces.Discrete(self.total_actions)

    def action(self, flat_action: int) -> np.ndarray:
        """Decode flat -> reduced multi-action -> full 6-dim action."""
        reduced = np.zeros(len(self.reduced_nvec), dtype=np.int64)
        remaining = flat_action
        for i in range(len(self.reduced_nvec) - 1, -1, -1):
            reduced[i] = remaining % self.reduced_nvec[i]
            remaining //= self.reduced_nvec[i]
        # Reinsert neutral roll at position _ROLL_IDX
        full = np.insert(reduced, self._ROLL_IDX, self._ROLL_NEUTRAL).astype(np.int64)
        return full

    def reverse_action(self, multi_action: np.ndarray) -> int:
        """Encode full 6-dim action -> flat (drops roll)."""
        reduced = np.delete(multi_action, self._ROLL_IDX)
        flat = 0
        multiplier = 1
        for i in range(len(self.reduced_nvec) - 1, -1, -1):
            flat += int(reduced[i]) * multiplier
            multiplier *= self.reduced_nvec[i]
        return flat


In [8]:
%%writefile astro_env/astro_exploration_env.py
"""AstroExploration Gymnasium Environment."""

import numpy as np
import gymnasium as gym
from gymnasium import spaces
from astro_env.celestial_bodies import CELESTIAL_BODIES, TARGET_BODIES, INSTRUMENTS
from astro_env.physics import (
    gravitational_acceleration, orientation_to_direction,
    compute_orbital_position, normalize_distance, compute_heading,
)
from astro_env.rewards import RewardCalculator


class AstroExplorationEnv(gym.Env):
    """Custom Gymnasium environment for astrobiological exploration."""

    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 30}
    THRUST_LEVELS = [0.0, 0.25, 0.5, 0.75, 1.0]
    ROTATION_DELTAS = [-5.0, 0.0, 5.0]
    MAX_THRUST = 0.001
    FUEL_CONSUMPTION_RATE = 0.0005
    BATTERY_DRAIN_RATE = 0.00005
    SOLAR_RECHARGE_RANGE = 1.5
    SOLAR_RECHARGE_RATE = 0.0001
    DT = 1.0
    SNR_DETECTION_THRESHOLD = 0.5
    BIOSIG_SUCCESS_COUNT = 3
    MAX_DISTANCE = 50.0

    def __init__(
        self,
        render_mode=None,
        max_episode_steps=100_000,
        enable_curriculum=False,
        success_count=3,
        detection_zone_scale=1.0,
        reward_kwargs=None,
    ):
        super().__init__()
        self.render_mode = render_mode
        self.max_episode_steps = max_episode_steps
        self.enable_curriculum = enable_curriculum
        self.curriculum_progress = 0.0
        self.success_count = success_count
        self.detection_zone_scale = detection_zone_scale

        obs_low = np.array(
            [-50, -50, -50, -10, -10, -10, 0, -1, -1, -1, 0, -1, -1, -1,
             0, -1, -1, -1, 0, 0, 0, 0, 0, 0, 0], dtype=np.float32)
        obs_high = np.array(
            [50, 50, 50, 10, 10, 10, 1, 1, 1, 1, 1, 1, 1, 1,
             1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=np.float32)
        self.observation_space = spaces.Box(obs_low, obs_high, dtype=np.float32)
        self.action_space = spaces.MultiDiscrete([5, 3, 3, 3, 4, 2])

        self.reward_calculator = RewardCalculator(**(reward_kwargs or {}))
        self._renderer = None

        self.base_initial_angles = {
            name: float(body["initial_angle"])
            for name, body in CELESTIAL_BODIES.items()
            if name != "Sun"
        }

        self.position = self.velocity = self.orientation = None
        self.fuel = self.battery = self.current_step = self.sim_time = None
        self.biosignatures_found = self.biosignatures_transmitted = None
        self.active_instrument = self.body_positions = None
        self.cumulative_reward = self.trajectory = None
        self.prev_min_target_distance = None
        self.prev_instrument = None
        self.instrument_hold_steps = 0
        self.phase = "cruise"
        self.phase_steps = 0

    def set_curriculum_progress(self, p):
        self.curriculum_progress = float(np.clip(p, 0.0, 1.0))

    def _scaled_zone_radius(self, target_name):
        return CELESTIAL_BODIES[target_name]["detection_zone_radius"] * self.detection_zone_scale

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        earth_data = CELESTIAL_BODIES["Earth"]
        self.position = np.array([earth_data["orbit_radius"], 0.0, 0.0], dtype=np.float64)
        v_earth = 2.0 * np.pi * earth_data["orbit_radius"] / earth_data["orbit_period"]
        self.velocity = np.array([0.0, v_earth, 0.0], dtype=np.float64)
        self.orientation = np.array([0.0, 0.0, 0.0], dtype=np.float64)
        self.fuel = 1.0
        self.battery = 1.0
        self.current_step = 0
        self.sim_time = 0.0
        self.biosignatures_found = set()
        self.biosignatures_transmitted = set()
        self.active_instrument = 0
        self.prev_instrument = 0
        self.instrument_hold_steps = 0
        self.phase = "cruise"
        self.phase_steps = 0
        self.cumulative_reward = 0.0
        self.trajectory = [self.position.copy()]

        if self.np_random is not None:
            for name in self.base_initial_angles:
                CELESTIAL_BODIES[name]["initial_angle"] = self.base_initial_angles[name] + self.np_random.uniform(-0.08, 0.08)

        self._update_body_positions()
        self.prev_min_target_distance = self._min_target_distance()
        return self._get_obs(), self._get_info()

    def _nearest_target(self):
        best_name = TARGET_BODIES[0]
        best_dist = float("inf")
        for t in TARGET_BODIES:
            d = np.linalg.norm(self.body_positions.get(t, np.zeros(3)) - self.position)
            if d < best_dist:
                best_dist = d
                best_name = t
        return best_name, best_dist

    def _recommended_instruments(self, target_name):
        biosigs = CELESTIAL_BODIES[target_name]["biosignatures"]
        compatible = []
        for idx, item in INSTRUMENTS.items():
            if idx == 0:
                continue
            if any(b in item["detects"] for b in biosigs):
                compatible.append(idx)
        return compatible

    def _update_phase(self, min_target_dist, in_zone, has_new_detection, transmitted):
        if self.phase == "cruise" and min_target_dist < 8.0:
            self.phase = "approach"
            self.phase_steps = 0
        elif self.phase == "approach" and in_zone:
            self.phase = "detect"
            self.phase_steps = 0
        elif self.phase == "detect" and (has_new_detection or len(self.biosignatures_found) > 0):
            self.phase = "transmit"
            self.phase_steps = 0
        elif self.phase == "transmit" and transmitted:
            self.phase = "approach"
            self.phase_steps = 0
        self.phase_steps += 1

    def step(self, action):
        assert self.action_space.contains(action)
        thrust_idx, pitch_idx, yaw_idx, roll_idx, instrument_idx, comm_idx = action

        thrust_level = self.THRUST_LEVELS[thrust_idx]
        self.orientation[0] += np.radians(self.ROTATION_DELTAS[pitch_idx])
        self.orientation[1] += np.radians(self.ROTATION_DELTAS[yaw_idx])
        self.orientation[2] += np.radians(self.ROTATION_DELTAS[roll_idx])
        self.orientation = np.clip(self.orientation, -np.pi, np.pi)

        switched = instrument_idx != self.prev_instrument
        if switched:
            self.instrument_hold_steps = 0
        else:
            self.instrument_hold_steps += 1
        self.active_instrument = instrument_idx

        fuel_used = 0.0
        if thrust_level > 0 and self.fuel > 0:
            forward = orientation_to_direction(*self.orientation)
            self.velocity += forward * thrust_level * self.MAX_THRUST * self.DT
            fuel_used = thrust_level * self.FUEL_CONSUMPTION_RATE
            self.fuel = max(0.0, self.fuel - fuel_used)

        total_grav = np.zeros(3)
        for name, body in CELESTIAL_BODIES.items():
            body_pos = np.zeros(3) if name == "Sun" else self.body_positions[name]
            total_grav += gravitational_acceleration(self.position, body_pos, body["mass"])
        self.velocity += total_grav * self.DT
        self.velocity = np.clip(self.velocity, -10.0, 10.0)
        self.position += self.velocity * self.DT

        self.sim_time += self.DT
        self._update_body_positions()

        self.battery -= self.BATTERY_DRAIN_RATE
        if np.linalg.norm(self.position) < self.SOLAR_RECHARGE_RANGE:
            self.battery += self.SOLAR_RECHARGE_RATE
        self.battery = np.clip(self.battery, 0.0, 1.0)

        nearest_target, min_target_dist = self._nearest_target()
        zone_radius = self._scaled_zone_radius(nearest_target)
        in_zone = min_target_dist < zone_radius
        recommended = self._recommended_instruments(nearest_target)
        instrument_compatible = instrument_idx in recommended
        switched_instrument_far = switched and (min_target_dist > 2.0)

        snr = self._compute_snr()
        new_biosigs = []
        if instrument_idx > 0 and snr >= self.SNR_DETECTION_THRESHOLD:
            new_biosigs = self._attempt_detection(instrument_idx)

        new_tx = []
        if comm_idx == 1:
            for b in list(self.biosignatures_found):
                if b not in self.biosignatures_transmitted:
                    self.biosignatures_transmitted.add(b)
                    new_tx.append(b)

        orbital_insertion = self._check_orbital_insertion()
        collision = self._check_collision()
        out_of_bounds = np.linalg.norm(self.position) > self.MAX_DISTANCE
        resource_depleted = self.fuel <= 0 or self.battery <= 0
        success = len(self.biosignatures_transmitted) >= self.success_count

        self.current_step += 1
        # Cap unproductive episodes earlier to improve signal.
        horizon = int(self.max_episode_steps if not self.enable_curriculum else max(3000, 10000 - 7000 * self.curriculum_progress))
        truncated = self.current_step >= horizon
        terminated = collision or out_of_bounds or resource_depleted or success

        self._update_phase(min_target_dist, in_zone, len(new_biosigs) > 0, len(new_tx) > 0)

        reward_state = {
            "fuel_used": fuel_used,
            "collision": collision,
            "out_of_bounds": out_of_bounds,
            "new_biosignatures": new_biosigs,
            "new_transmissions": new_tx,
            "orbital_insertion": orbital_insertion,
            "min_target_distance": min_target_dist,
            "prev_min_target_distance": self.prev_min_target_distance,
            "in_detection_zone": in_zone,
            "instrument_compatible": instrument_compatible,
            "instrument_hold_steps": self.instrument_hold_steps,
            "switched_instrument_far": switched_instrument_far,
            "success": success,
        }
        reward, reward_info = self.reward_calculator.compute(reward_state)

        # Lightweight mission-flow shaping.
        if self.phase == "approach":
            reward += 0.1
            reward_info["reward_phase_approach"] = 0.1
        elif self.phase == "detect" and instrument_compatible:
            reward += 0.2
            reward_info["reward_phase_detect"] = 0.2
        elif self.phase == "transmit" and comm_idx == 1:
            reward += 0.2
            reward_info["reward_phase_transmit"] = 0.2

        self.cumulative_reward += reward
        self.trajectory.append(self.position.copy())
        if len(self.trajectory) > 500:
            self.trajectory.pop(0)

        self.prev_min_target_distance = min_target_dist
        self.prev_instrument = instrument_idx

        obs = self._get_obs()
        info = self._get_info()
        info.update(reward_info)
        info["success"] = success
        info["collision"] = collision
        info["out_of_bounds"] = out_of_bounds
        info["resource_depleted"] = resource_depleted
        info["min_target_distance"] = min_target_dist
        info["nearest_target"] = nearest_target
        info["in_detection_zone"] = in_zone
        info["instrument_compatible"] = instrument_compatible
        info["phase"] = self.phase
        info["detected_count"] = len(new_biosigs)
        info["transmitted_count"] = len(new_tx)
        return obs, reward, terminated, truncated, info

    def _update_body_positions(self):
        self.body_positions = {}
        for name, body in CELESTIAL_BODIES.items():
            if name == "Sun":
                self.body_positions[name] = np.zeros(3)
            elif body["parent"] == "Sun":
                self.body_positions[name] = compute_orbital_position(
                    body["orbit_radius"], body["orbit_period"],
                    self.sim_time, body["initial_angle"])
        for name, body in CELESTIAL_BODIES.items():
            if body["parent"] and body["parent"] != "Sun":
                parent_pos = self.body_positions.get(body["parent"], np.zeros(3))
                self.body_positions[name] = compute_orbital_position(
                    body["orbit_radius"], body["orbit_period"],
                    self.sim_time, body["initial_angle"], parent_pos=parent_pos)

    def _get_obs(self):
        obs = np.zeros(25, dtype=np.float32)
        obs[0:3] = self.position.astype(np.float32)
        obs[3:6] = np.clip(self.velocity, -10, 10).astype(np.float32)
        for i, target_name in enumerate(TARGET_BODIES):
            target_pos = self.body_positions.get(target_name, np.zeros(3))
            dist = np.linalg.norm(target_pos - self.position)
            heading = compute_heading(self.position, target_pos)
            base_idx = 6 + i * 4
            obs[base_idx] = normalize_distance(dist)
            obs[base_idx + 1:base_idx + 4] = heading.astype(np.float32)
        obs[18] = self.fuel
        obs[19] = self.battery
        obs[20] = self._compute_snr()
        obs[21] = len(self.biosignatures_found) / max(1, self.success_count)
        obs[22] = len(self.biosignatures_transmitted) / max(1, self.success_count)

        nearest_target, _ = self._nearest_target()
        target_idx = TARGET_BODIES.index(nearest_target)
        obs[23] = target_idx / max(1, len(TARGET_BODIES) - 1)
        obs[24] = 1.0 if self.active_instrument in self._recommended_instruments(nearest_target) else 0.0
        return obs

    def _get_info(self):
        return {
            "step": self.current_step,
            "fuel": self.fuel,
            "battery": self.battery,
            "position": self.position.copy(),
            "velocity": self.velocity.copy(),
            "biosig_found": list(self.biosignatures_found),
            "biosig_transmitted": list(self.biosignatures_transmitted),
            "cumulative_reward": self.cumulative_reward,
            "active_instrument": INSTRUMENTS[self.active_instrument]["name"],
        }

    def _compute_snr(self):
        max_snr = 0.0
        for target_name in TARGET_BODIES:
            body = CELESTIAL_BODIES[target_name]
            target_pos = self.body_positions.get(target_name, np.zeros(3))
            dist = np.linalg.norm(target_pos - self.position)
            zone_radius = body["detection_zone_radius"] * self.detection_zone_scale
            if zone_radius > 0 and dist < zone_radius:
                max_snr = max(max_snr, 1.0 - (dist / zone_radius))
        return float(max_snr)

    def _attempt_detection(self, instrument_idx):
        detected = []
        instrument = INSTRUMENTS[instrument_idx]
        for target_name in TARGET_BODIES:
            body = CELESTIAL_BODIES[target_name]
            target_pos = self.body_positions.get(target_name, np.zeros(3))
            dist = np.linalg.norm(target_pos - self.position)
            if dist < body["detection_zone_radius"] * self.detection_zone_scale:
                for biosig in body["biosignatures"]:
                    if biosig in instrument["detects"] and biosig not in self.biosignatures_found:
                        self.biosignatures_found.add(biosig)
                        detected.append(biosig)
        return detected

    def _check_collision(self):
        for name, body in CELESTIAL_BODIES.items():
            body_pos = self.body_positions.get(name, np.zeros(3))
            dist = np.linalg.norm(body_pos - self.position)
            if dist < max(body["radius"] * 10, 0.001):
                return True
        return False

    def _check_orbital_insertion(self):
        for target_name in TARGET_BODIES:
            target_pos = self.body_positions.get(target_name, np.zeros(3))
            dist = np.linalg.norm(target_pos - self.position)
            if (dist < self._scaled_zone_radius(target_name) * 0.5
                    and np.linalg.norm(self.velocity) < 0.01):
                return True
        return False

    def _min_target_distance(self):
        return min(
            np.linalg.norm(self.body_positions.get(t, np.zeros(3)) - self.position)
            for t in TARGET_BODIES)

    def render(self):
        if self.render_mode is None:
            return None
        if self._renderer is None:
            from visualization.renderer import PygameRenderer
            self._renderer = PygameRenderer(self)
        return self._renderer.render_frame(self._get_render_state())

    def _get_render_state(self):
        return {
            "position": self.position.copy(), "velocity": self.velocity.copy(),
            "orientation": self.orientation.copy(), "fuel": self.fuel,
            "battery": self.battery, "snr": self._compute_snr(),
            "active_instrument": INSTRUMENTS[self.active_instrument]["name"],
            "biosig_found": list(self.biosignatures_found),
            "biosig_transmitted": list(self.biosignatures_transmitted),
            "body_positions": {k: v.copy() for k, v in self.body_positions.items()},
            "trajectory": [p.copy() for p in self.trajectory],
            "current_step": self.current_step, "max_steps": self.max_episode_steps,
            "cumulative_reward": self.cumulative_reward, "thrust_level": 0.0,
            "phase": self.phase,
        }

    def close(self):
        if self._renderer is not None:
            self._renderer.close()
            self._renderer = None


In [11]:
%%writefile agents/__init__.py


In [12]:
%%writefile agents/reinforce_policy.py
"""PyTorch policy network for REINFORCE algorithm."""

import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical


class REINFORCEPolicy(nn.Module):
    """Multi-head policy for MultiDiscrete action space."""

    def __init__(self, obs_dim=23, action_nvec=None, hidden_sizes=None):
        super().__init__()
        if action_nvec is None:
            action_nvec = [5, 3, 3, 3, 4, 2]
        if hidden_sizes is None:
            hidden_sizes = [128, 64]
        self.action_nvec = action_nvec
        layers, prev_size = [], obs_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev_size, h), nn.ReLU()]
            prev_size = h
        self.backbone = nn.Sequential(*layers)
        self.heads = nn.ModuleList([nn.Linear(prev_size, n) for n in action_nvec])

    def forward(self, obs):
        features = self.backbone(obs)
        return [head(features) for head in self.heads]

    def get_action(self, obs):
        logits_list = self.forward(obs)
        actions, total_log_prob = [], torch.tensor(0.0)
        for logits in logits_list:
            dist = Categorical(logits=logits)
            action = dist.sample()
            total_log_prob = total_log_prob + dist.log_prob(action)
            actions.append(action.item())
        return np.array(actions, dtype=np.int64), total_log_prob

    def evaluate_actions(self, obs, actions):
        logits_list = self.forward(obs)
        total_log_prob = torch.zeros(obs.shape[0])
        for i, logits in enumerate(logits_list):
            dist = Categorical(logits=logits)
            total_log_prob = total_log_prob + dist.log_prob(actions[:, i])
        return total_log_prob


In [13]:
%%writefile agents/reinforce_agent.py
"""REINFORCE (Monte Carlo Policy Gradient) training agent."""

import csv
import os
import numpy as np
import torch
import torch.optim as optim
from agents.reinforce_policy import REINFORCEPolicy


class REINFORCEAgent:
    """REINFORCE with optional baseline subtraction and entropy regularization."""

    def __init__(
        self,
        env,
        learning_rate=3e-4,
        gamma=0.99,
        hidden_sizes=None,
        baseline="mean",
        entropy_coef=0.01,
        grad_clip_norm=1.0,
        seed=42,
    ):
        self.env = env
        self.gamma = gamma
        self.baseline = baseline
        self.entropy_coef = entropy_coef
        self.grad_clip_norm = grad_clip_norm

        torch.manual_seed(seed)
        np.random.seed(seed)

        self.policy = REINFORCEPolicy(
            obs_dim=env.observation_space.shape[0],
            action_nvec=list(env.action_space.nvec),
            hidden_sizes=hidden_sizes or [256, 128],
        )
        self.optimizer = optim.Adam(self.policy.parameters(), lr=learning_rate)

        self.running_return = 0.0
        self.running_count = 0
        self.kpi = {
            "episode_reward": [],
            "episode_length": [],
            "detected_total": [],
            "transmitted_total": [],
            "min_distance": [],
            "success": [],
            "compatible_rate": [],
            "phase_detect_steps": [],
        }

    def collect_episode(self):
        obs, _ = self.env.reset()
        log_probs, rewards, entropies = [], [], []
        done = False

        detected_total = 0
        transmitted_total = 0
        min_distance = 1e9
        compatible_steps = 0
        detect_phase_steps = 0
        steps = 0

        while not done:
            obs_tensor = torch.FloatTensor(obs)
            logits_list = self.policy.forward(obs_tensor)

            action_items = []
            total_log_prob = torch.tensor(0.0)
            total_entropy = torch.tensor(0.0)
            for logits in logits_list:
                dist = torch.distributions.Categorical(logits=logits)
                action = dist.sample()
                total_log_prob = total_log_prob + dist.log_prob(action)
                total_entropy = total_entropy + dist.entropy()
                action_items.append(action.item())

            action = np.array(action_items, dtype=np.int64)
            obs, reward, terminated, truncated, info = self.env.step(action)

            log_probs.append(total_log_prob)
            entropies.append(total_entropy)
            rewards.append(reward)

            done = terminated or truncated
            steps += 1

            detected_total += int(info.get("detected_count", 0))
            transmitted_total += int(info.get("transmitted_count", 0))
            min_distance = min(min_distance, float(info.get("min_target_distance", 1e9)))
            compatible_steps += int(bool(info.get("instrument_compatible", False)))
            detect_phase_steps += int(info.get("phase", "") == "detect")

        metrics = {
            "detected_total": detected_total,
            "transmitted_total": transmitted_total,
            "min_distance": min_distance if min_distance < 1e8 else 0.0,
            "success": int(bool(info.get("success", False))),
            "compatible_rate": compatible_steps / max(1, steps),
            "phase_detect_steps": detect_phase_steps,
        }
        return log_probs, rewards, entropies, len(rewards), metrics

    def compute_returns(self, rewards):
        returns, G = [], 0.0
        for r in reversed(rewards):
            G = r + self.gamma * G
            returns.insert(0, G)
        returns = torch.FloatTensor(returns)

        if self.baseline == "mean":
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        elif self.baseline == "running":
            self.running_count += 1
            self.running_return = 0.95 * self.running_return + 0.05 * returns.mean().item()
            returns = returns - self.running_return
        elif self.baseline == "none":
            pass

        # Advantage normalization helps reduce variance.
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        return returns

    def update(self, log_probs, returns, entropies):
        pg_loss = torch.tensor(0.0)
        ent_loss = torch.tensor(0.0)
        for lp, G, ent in zip(log_probs, returns, entropies):
            pg_loss = pg_loss + (-lp * G)
            ent_loss = ent_loss + ent

        pg_loss = pg_loss / len(log_probs)
        ent_loss = ent_loss / len(entropies)
        total_loss = pg_loss - self.entropy_coef * ent_loss

        self.optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy.parameters(), self.grad_clip_norm)
        self.optimizer.step()

        return total_loss.item(), pg_loss.item(), ent_loss.item()

    def train(self, num_episodes=5000, log_interval=100, save_dir=None, curriculum_episodes=500):
        history, reward_window = [], []

        for episode in range(1, num_episodes + 1):
            # Curriculum control signal for env.
            if hasattr(self.env.unwrapped, "set_curriculum_progress"):
                progress = min(1.0, episode / max(1, curriculum_episodes))
                self.env.unwrapped.set_curriculum_progress(progress)

            log_probs, rewards, entropies, ep_len, metrics = self.collect_episode()
            ep_reward = float(sum(rewards))
            returns = self.compute_returns(rewards)
            total_loss, pg_loss, ent = self.update(log_probs, returns, entropies)

            history.append((ep_reward, ep_len))
            reward_window.append(ep_reward)
            if len(reward_window) > 100:
                reward_window.pop(0)

            self.kpi["episode_reward"].append(ep_reward)
            self.kpi["episode_length"].append(ep_len)
            self.kpi["detected_total"].append(metrics["detected_total"])
            self.kpi["transmitted_total"].append(metrics["transmitted_total"])
            self.kpi["min_distance"].append(metrics["min_distance"])
            self.kpi["success"].append(metrics["success"])
            self.kpi["compatible_rate"].append(metrics["compatible_rate"])
            self.kpi["phase_detect_steps"].append(metrics["phase_detect_steps"])

            if episode % log_interval == 0:
                print(
                    f"Episode {episode:5d} | Avg Reward: {np.mean(reward_window):10.2f} | "
                    f"Avg Length: {np.mean([h[1] for h in history[-100:]]):8.1f} | "
                    f"Loss: {total_loss:9.4f} | PG: {pg_loss:8.4f} | Ent: {ent:7.3f}"
                )
                print(
                    f"   KPI: success={np.mean(self.kpi['success'][-100:]):.2f}, "
                    f"detect={np.mean(self.kpi['detected_total'][-100:]):.2f}, "
                    f"transmit={np.mean(self.kpi['transmitted_total'][-100:]):.2f}, "
                    f"min_dist={np.mean(self.kpi['min_distance'][-100:]):.3f}, "
                    f"compat={np.mean(self.kpi['compatible_rate'][-100:]):.2f}"
                )

        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
            torch.save(self.policy.state_dict(), os.path.join(save_dir, "policy.pt"))

            rewards_path = os.path.join(save_dir, "rewards.csv")
            with open(rewards_path, "w", newline="") as f:
                w = csv.writer(f)
                w.writerow(["episode", "reward", "episode_length", "success", "detected", "transmitted", "min_distance", "compatible_rate"])
                for i, (r, l) in enumerate(history, 1):
                    w.writerow([
                        i,
                        r,
                        l,
                        self.kpi["success"][i - 1],
                        self.kpi["detected_total"][i - 1],
                        self.kpi["transmitted_total"][i - 1],
                        self.kpi["min_distance"][i - 1],
                        self.kpi["compatible_rate"][i - 1],
                    ])

            print(f"Model saved to {save_dir}/policy.pt")

        return history, self.kpi


In [14]:
%%writefile training/hyperparams.py
"""Hyperparameter configurations for all 30 experiments.

10 configurations per algorithm, each varying 1-2 parameters from a baseline.
"""

# Total training timesteps for SB3 algorithms
TOTAL_TIMESTEPS = 500_000
EVAL_FREQ = 10_000
N_EVAL_EPISODES = 5

# REINFORCE training episodes
REINFORCE_EPISODES = 5000

# ============================================================
# Shared DQN environment kwargs
# - success_count=1: only need 1 transmitted biosignature to win (easier early goal)
# - detection_zone_scale=2.0: larger target zones make early detections reachable
# - enable_curriculum=True: episode horizon shrinks from 10K -> 3K as agent improves
# - reward_kwargs: rebalanced so collision (-80) no longer drowns out success (+2000)
# ============================================================
_DQN_ENV_KWARGS = {
    "enable_curriculum": True,
    "success_count": 1,
    "detection_zone_scale": 2.0,
    "reward_kwargs": {
        "step_fuel_penalty": 0.001,
        "step_time_penalty": 0.0001,
        "direction_bonus_scale": 4.0,
        "moving_away_penalty": 0.05,
        "compatible_instrument_bonus": 8.0,
        "in_zone_hold_bonus": 2.0,
        "proximity_scale": 1.0,
        "transmission_bonus": 600.0,
        "success_completion_bonus": 2000.0,
        "collision_penalty": -80.0,
    },
}

# ============================================================
# DQN Configurations (10 runs)
# All runs share _DQN_ENV_KWARGS so reward balance and zone scale are consistent.
# use_per=True enables PrioritizedReplayBuffer (sb3-contrib) — graceful fallback
#   if sb3-contrib is absent.
# n_steps>1 propagates sparse rewards back N steps (SB3 DQN supports this).
# exploration_fraction >= 0.6 because Discrete(360) still needs wide coverage.
# ============================================================
DQN_CONFIGS = [
    {   # Run 0: Baseline + PER
        "name": "dqn_baseline",
        "learning_rate": 1e-4,
        "buffer_size": 100_000,
        "batch_size": 64,
        "tau": 1.0,
        "gamma": 0.995,
        "exploration_fraction": 0.6,
        "exploration_final_eps": 0.05,
        "learning_starts": 3000,
        "policy_kwargs": {"net_arch": [256, 256]},
        "use_per": True,
        "per_alpha": 0.6,
        "per_beta": 0.4,
        "env_kwargs": _DQN_ENV_KWARGS,
    },
    {   # Run 1: Smaller network + PER
        "name": "dqn_small_net",
        "learning_rate": 1e-4,
        "buffer_size": 100_000,
        "batch_size": 64,
        "tau": 1.0,
        "gamma": 0.995,
        "exploration_fraction": 0.6,
        "exploration_final_eps": 0.05,
        "learning_starts": 3000,
        "policy_kwargs": {"net_arch": [128, 128]},
        "use_per": True,
        "per_alpha": 0.6,
        "per_beta": 0.4,
        "env_kwargs": _DQN_ENV_KWARGS,
    },
    {   # Run 2: Higher learning rate + PER
        "name": "dqn_high_lr",
        "learning_rate": 5e-4,
        "buffer_size": 100_000,
        "batch_size": 64,
        "tau": 1.0,
        "gamma": 0.995,
        "exploration_fraction": 0.6,
        "exploration_final_eps": 0.05,
        "learning_starts": 3000,
        "policy_kwargs": {"net_arch": [256, 256]},
        "use_per": True,
        "per_alpha": 0.6,
        "per_beta": 0.4,
        "env_kwargs": _DQN_ENV_KWARGS,
    },
    {   # Run 3: Lower learning rate + PER
        "name": "dqn_low_lr",
        "learning_rate": 1e-5,
        "buffer_size": 100_000,
        "batch_size": 64,
        "tau": 1.0,
        "gamma": 0.995,
        "exploration_fraction": 0.6,
        "exploration_final_eps": 0.05,
        "learning_starts": 3000,
        "policy_kwargs": {"net_arch": [256, 256]},
        "use_per": True,
        "per_alpha": 0.6,
        "per_beta": 0.4,
        "env_kwargs": _DQN_ENV_KWARGS,
    },
    {   # Run 4: Large buffer + PER with high alpha (strong prioritization)
        "name": "dqn_large_buffer",
        "learning_rate": 1e-4,
        "buffer_size": 300_000,
        "batch_size": 128,
        "tau": 1.0,
        "gamma": 0.995,
        "exploration_fraction": 0.65,
        "exploration_final_eps": 0.02,
        "learning_starts": 3000,
        "policy_kwargs": {"net_arch": [256, 256]},
        "use_per": True,
        "per_alpha": 0.7,
        "per_beta": 0.4,
        "env_kwargs": _DQN_ENV_KWARGS,
    },
    {   # Run 5: Soft target update + PER
        "name": "dqn_soft_update",
        "learning_rate": 1e-4,
        "buffer_size": 100_000,
        "batch_size": 64,
        "tau": 0.005,
        "gamma": 0.995,
        "exploration_fraction": 0.6,
        "exploration_final_eps": 0.05,
        "learning_starts": 3000,
        "policy_kwargs": {"net_arch": [256, 256]},
        "use_per": True,
        "per_alpha": 0.6,
        "per_beta": 0.4,
        "env_kwargs": _DQN_ENV_KWARGS,
    },
    {   # Run 6: Lower gamma (prioritises near-term navigation rewards)
        "name": "dqn_low_gamma",
        "learning_rate": 1e-4,
        "buffer_size": 100_000,
        "batch_size": 64,
        "tau": 1.0,
        "gamma": 0.95,
        "exploration_fraction": 0.65,
        "exploration_final_eps": 0.05,
        "learning_starts": 3000,
        "policy_kwargs": {"net_arch": [256, 256]},
        "use_per": True,
        "per_alpha": 0.6,
        "per_beta": 0.4,
        "env_kwargs": _DQN_ENV_KWARGS,
    },
    {   # Run 7: Deeper network + PER
        "name": "dqn_deep_net",
        "learning_rate": 1e-4,
        "buffer_size": 100_000,
        "batch_size": 64,
        "tau": 1.0,
        "gamma": 0.995,
        "exploration_fraction": 0.6,
        "exploration_final_eps": 0.05,
        "learning_starts": 3000,
        "policy_kwargs": {"net_arch": [256, 256, 128]},
        "use_per": True,
        "per_alpha": 0.6,
        "per_beta": 0.4,
        "env_kwargs": _DQN_ENV_KWARGS,
    },
    {   # Run 8: Medium lr + large batch + PER
        "name": "dqn_med_lr_batch",
        "learning_rate": 3e-4,
        "buffer_size": 200_000,
        "batch_size": 256,
        "tau": 1.0,
        "gamma": 0.995,
        "exploration_fraction": 0.6,
        "exploration_final_eps": 0.05,
        "learning_starts": 3000,
        "policy_kwargs": {"net_arch": [256, 256]},
        "use_per": True,
        "per_alpha": 0.6,
        "per_beta": 0.4,
        "env_kwargs": _DQN_ENV_KWARGS,
    },
    {   # Run 9: Long exploration + very high PER alpha (maximum exploitation of rare wins)
        "name": "dqn_long_explore",
        "learning_rate": 1e-4,
        "buffer_size": 100_000,
        "batch_size": 64,
        "tau": 1.0,
        "gamma": 0.995,
        "exploration_fraction": 0.75,
        "exploration_final_eps": 0.01,
        "learning_starts": 3000,
        "policy_kwargs": {"net_arch": [256, 256]},
        "use_per": True,
        "per_alpha": 0.8,
        "per_beta": 0.4,
        "env_kwargs": _DQN_ENV_KWARGS,
    },
]

# ============================================================
# PPO Configurations (10 runs)
# Baseline: lr=3e-4, n_steps=2048, batch=64, n_epochs=10,
#           clip=0.2, ent=0.01, gamma=0.99, net=[256,256]
# ============================================================
PPO_CONFIGS = [
    {   # Run 0: Baseline
        "name": "ppo_baseline",
        "learning_rate": 3e-4,
        "n_steps": 2048,
        "batch_size": 64,
        "n_epochs": 10,
        "clip_range": 0.2,
        "ent_coef": 0.01,
        "vf_coef": 0.5,
        "gamma": 0.99,
        "gae_lambda": 0.95,
        "max_grad_norm": 0.5,
        "policy_kwargs": {"net_arch": [256, 256]},
    },
    {   # Run 1: Higher entropy
        "name": "ppo_high_entropy",
        "learning_rate": 3e-4,
        "n_steps": 2048,
        "batch_size": 64,
        "n_epochs": 10,
        "clip_range": 0.2,
        "ent_coef": 0.05,
        "vf_coef": 0.5,
        "gamma": 0.99,
        "gae_lambda": 0.95,
        "max_grad_norm": 0.5,
        "policy_kwargs": {"net_arch": [256, 256]},
    },
    {   # Run 2: Lower learning rate
        "name": "ppo_low_lr",
        "learning_rate": 1e-4,
        "n_steps": 2048,
        "batch_size": 64,
        "n_epochs": 10,
        "clip_range": 0.2,
        "ent_coef": 0.01,
        "vf_coef": 0.5,
        "gamma": 0.99,
        "gae_lambda": 0.95,
        "max_grad_norm": 0.5,
        "policy_kwargs": {"net_arch": [256, 256]},
    },
    {   # Run 3: Tighter clipping
        "name": "ppo_tight_clip",
        "learning_rate": 3e-4,
        "n_steps": 2048,
        "batch_size": 64,
        "n_epochs": 10,
        "clip_range": 0.1,
        "ent_coef": 0.01,
        "vf_coef": 0.5,
        "gamma": 0.99,
        "gae_lambda": 0.95,
        "max_grad_norm": 0.5,
        "policy_kwargs": {"net_arch": [256, 256]},
    },
    {   # Run 4: Wider clipping
        "name": "ppo_wide_clip",
        "learning_rate": 3e-4,
        "n_steps": 2048,
        "batch_size": 64,
        "n_epochs": 10,
        "clip_range": 0.3,
        "ent_coef": 0.01,
        "vf_coef": 0.5,
        "gamma": 0.99,
        "gae_lambda": 0.95,
        "max_grad_norm": 0.5,
        "policy_kwargs": {"net_arch": [256, 256]},
    },
    {   # Run 5: More epochs
        "name": "ppo_more_epochs",
        "learning_rate": 3e-4,
        "n_steps": 2048,
        "batch_size": 64,
        "n_epochs": 20,
        "clip_range": 0.2,
        "ent_coef": 0.01,
        "vf_coef": 0.5,
        "gamma": 0.99,
        "gae_lambda": 0.95,
        "max_grad_norm": 0.5,
        "policy_kwargs": {"net_arch": [256, 256]},
    },
    {   # Run 6: Smaller network
        "name": "ppo_small_net",
        "learning_rate": 3e-4,
        "n_steps": 2048,
        "batch_size": 64,
        "n_epochs": 10,
        "clip_range": 0.2,
        "ent_coef": 0.01,
        "vf_coef": 0.5,
        "gamma": 0.99,
        "gae_lambda": 0.95,
        "max_grad_norm": 0.5,
        "policy_kwargs": {"net_arch": [128, 128]},
    },
    {   # Run 7: Larger rollout + batch
        "name": "ppo_large_rollout",
        "learning_rate": 3e-4,
        "n_steps": 4096,
        "batch_size": 128,
        "n_epochs": 10,
        "clip_range": 0.2,
        "ent_coef": 0.01,
        "vf_coef": 0.5,
        "gamma": 0.99,
        "gae_lambda": 0.95,
        "max_grad_norm": 0.5,
        "policy_kwargs": {"net_arch": [256, 256]},
    },
    {   # Run 8: Lower gamma
        "name": "ppo_low_gamma",
        "learning_rate": 3e-4,
        "n_steps": 2048,
        "batch_size": 64,
        "n_epochs": 10,
        "clip_range": 0.2,
        "ent_coef": 0.01,
        "vf_coef": 0.5,
        "gamma": 0.95,
        "gae_lambda": 0.95,
        "max_grad_norm": 0.5,
        "policy_kwargs": {"net_arch": [256, 256]},
    },
    {   # Run 9: Higher lr + deeper network
        "name": "ppo_high_lr_deep",
        "learning_rate": 5e-4,
        "n_steps": 2048,
        "batch_size": 64,
        "n_epochs": 10,
        "clip_range": 0.2,
        "ent_coef": 0.01,
        "vf_coef": 0.5,
        "gamma": 0.99,
        "gae_lambda": 0.95,
        "max_grad_norm": 0.5,
        "policy_kwargs": {"net_arch": [256, 256, 128]},
    },
]

# ============================================================
# REINFORCE Configurations (10 runs)
# Baseline: lr=1e-3, gamma=0.99, hidden=[128,64], baseline=mean
# ============================================================
REINFORCE_CONFIGS = [
    {   # Run 0: Baseline
        "name": "reinforce_baseline",
        "learning_rate": 1e-3,
        "gamma": 0.99,
        "hidden_sizes": [128, 64],
        "baseline": "mean",
    },
    {   # Run 1: No baseline
        "name": "reinforce_no_baseline",
        "learning_rate": 1e-3,
        "gamma": 0.99,
        "hidden_sizes": [128, 64],
        "baseline": "none",
    },
    {   # Run 2: Lower learning rate
        "name": "reinforce_low_lr",
        "learning_rate": 1e-4,
        "gamma": 0.99,
        "hidden_sizes": [128, 64],
        "baseline": "mean",
    },
    {   # Run 3: Higher learning rate
        "name": "reinforce_high_lr",
        "learning_rate": 5e-3,
        "gamma": 0.99,
        "hidden_sizes": [128, 64],
        "baseline": "mean",
    },
    {   # Run 4: Larger network
        "name": "reinforce_large_net",
        "learning_rate": 1e-3,
        "gamma": 0.99,
        "hidden_sizes": [256, 128],
        "baseline": "mean",
    },
    {   # Run 5: Smaller network
        "name": "reinforce_small_net",
        "learning_rate": 1e-3,
        "gamma": 0.99,
        "hidden_sizes": [64, 32],
        "baseline": "mean",
    },
    {   # Run 6: Lower gamma
        "name": "reinforce_low_gamma",
        "learning_rate": 1e-3,
        "gamma": 0.95,
        "hidden_sizes": [128, 64],
        "baseline": "mean",
    },
    {   # Run 7: Very low gamma
        "name": "reinforce_very_low_gamma",
        "learning_rate": 1e-3,
        "gamma": 0.90,
        "hidden_sizes": [128, 64],
        "baseline": "mean",
    },
    {   # Run 8: Deeper network
        "name": "reinforce_deep_net",
        "learning_rate": 1e-3,
        "gamma": 0.99,
        "hidden_sizes": [256, 128, 64],
        "baseline": "mean",
    },
    {   # Run 9: Running baseline
        "name": "reinforce_running_baseline",
        "learning_rate": 3e-4,
        "gamma": 0.99,
        "hidden_sizes": [128, 64],
        "baseline": "running",
    },
]


In [15]:
%%writefile training/__init__.py


In [16]:
%%writefile training/train_dqn.py
"""Train DQN agent on AstroExploration environment."""

import sys
import os
import argparse
import csv
import time

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import astro_env  # noqa: F401
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
from astro_env.wrappers import ReducedDiscreteWrapper
from training.hyperparams import DQN_CONFIGS, TOTAL_TIMESTEPS, EVAL_FREQ, N_EVAL_EPISODES

# Prioritized Experience Replay — requires sb3-contrib.
# Falls back to standard ReplayBuffer if not installed.
try:
    from sb3_contrib.common.buffers import PrioritizedReplayBuffer
    _HAS_PER = True
except ImportError:
    PrioritizedReplayBuffer = None
    _HAS_PER = False


# Keys that are NOT DQN constructor kwargs and must be stripped before passing to DQN().
_NON_MODEL_KEYS = {"name", "env_kwargs", "use_per", "per_alpha", "per_beta"}


class MissionKPITrainCallback(BaseCallback):
    """Logs per-episode mission KPIs (success, detections, distance) to CSV."""

    def __init__(self, log_dir: str, verbose: int = 0):
        super().__init__(verbose)
        self.log_dir = log_dir
        self.rows = []

    def _on_step(self) -> bool:
        infos = self.locals.get("infos", [])
        dones = self.locals.get("dones", [])
        for info, done in zip(infos, dones):
            if done:
                self.rows.append({
                    "timesteps": int(self.num_timesteps),
                    "success": int(bool(info.get("success", False))),
                    "detected_count": int(info.get("detected_count", 0)),
                    "transmitted_count": int(info.get("transmitted_count", 0)),
                    "min_target_distance": float(info.get("min_target_distance", 0.0)),
                    "instrument_compatible": int(bool(info.get("instrument_compatible", False))),
                    "collision": int(bool(info.get("collision", False))),
                    "out_of_bounds": int(bool(info.get("out_of_bounds", False))),
                })
        return True

    def _on_training_end(self) -> None:
        if not self.rows:
            return
        os.makedirs(self.log_dir, exist_ok=True)
        path = os.path.join(self.log_dir, "kpi_train.csv")
        with open(path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(self.rows[0].keys()))
            writer.writeheader()
            writer.writerows(self.rows)


def _make_env(env_kwargs: dict) -> gym.Env:
    """Create and wrap AstroExploration env for DQN."""
    env = gym.make("AstroExploration-v0", **env_kwargs)
    return ReducedDiscreteWrapper(env)


def train_dqn(run_idx: int, seed: int = 42) -> dict:
    """Train a single DQN run with the given config index."""
    config = DQN_CONFIGS[run_idx]
    run_name = config["name"]
    print(f"\n{'='*60}")
    print(f"Training DQN - {run_name} (Run {run_idx})")
    print(f"{'='*60}")

    log_dir = f"results/logs/{run_name}"
    model_dir = f"results/models/{run_name}"
    os.makedirs(log_dir, exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)

    env_kwargs = config.get("env_kwargs", {})
    train_env = Monitor(_make_env(env_kwargs), log_dir)
    eval_env = Monitor(_make_env(env_kwargs))

    # Build DQN constructor kwargs — strip housekeeping keys.
    model_params = {k: v for k, v in config.items() if k not in _NON_MODEL_KEYS}

    # Wire up Prioritized Experience Replay when requested and available.
    use_per = config.get("use_per", False)
    if use_per and _HAS_PER:
        model_params["replay_buffer_class"] = PrioritizedReplayBuffer
        model_params["replay_buffer_kwargs"] = {
            "alpha": config.get("per_alpha", 0.6),
            "beta": config.get("per_beta", 0.4),
        }
        # PER is incompatible with optimize_memory_usage.
        model_params.pop("optimize_memory_usage", None)
        print(f"  PER enabled (alpha={config.get('per_alpha', 0.6)}, "
              f"beta={config.get('per_beta', 0.4)})")
    elif use_per and not _HAS_PER:
        print("  WARNING: use_per=True but sb3-contrib not installed. "
              "Falling back to standard ReplayBuffer.")

    model = DQN(
        "MlpPolicy",
        train_env,
        verbose=1,
        seed=seed,
        tensorboard_log=f"results/logs/{run_name}_tb",
        **model_params,
    )

    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=model_dir,
        log_path=log_dir,
        eval_freq=EVAL_FREQ,
        n_eval_episodes=N_EVAL_EPISODES,
        deterministic=True,
    )
    kpi_callback = MissionKPITrainCallback(log_dir)

    start = time.time()
    model.learn(
        total_timesteps=TOTAL_TIMESTEPS,
        callback=[eval_callback, kpi_callback],
        progress_bar=True,
    )
    wall_time = time.time() - start

    model.save(os.path.join(model_dir, "final_model"))
    print(f"\nTraining completed in {wall_time:.1f}s")
    print(f"Model saved to {model_dir}/final_model.zip")

    train_env.close()
    eval_env.close()

    return {"run_name": run_name, "wall_time": wall_time, "algorithm": "DQN"}


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Train DQN on AstroExploration")
    parser.add_argument("--run", type=int, default=0, help="Config index (0-9)")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    train_dqn(args.run, args.seed)


In [17]:
%%writefile training/train_ppo.py
"""Train PPO agent on AstroExploration environment."""

import os
import sys
import time
import argparse
import csv

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import astro_env  # noqa: F401
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
from training.hyperparams import PPO_CONFIGS, TOTAL_TIMESTEPS, EVAL_FREQ, N_EVAL_EPISODES


class MissionKPITrainCallback(BaseCallback):
    """Tracks mission KPIs from env info dict during training."""

    def __init__(self, log_dir, verbose=0):
        super().__init__(verbose)
        self.log_dir = log_dir
        self.rows = []

    def _on_step(self):
        infos = self.locals.get("infos", [])
        dones = self.locals.get("dones", [])
        if len(infos) == 0 or len(dones) == 0:
            return True

        for info, done in zip(infos, dones):
            if done:
                self.rows.append(
                    {
                        "timesteps": int(self.num_timesteps),
                        "success": int(bool(info.get("success", False))),
                        "detected_count": int(info.get("detected_count", 0)),
                        "transmitted_count": int(info.get("transmitted_count", 0)),
                        "min_target_distance": float(info.get("min_target_distance", 0.0)),
                        "instrument_compatible": int(bool(info.get("instrument_compatible", False))),
                    }
                )
        return True

    def _on_training_end(self):
        if not self.rows:
            return
        os.makedirs(self.log_dir, exist_ok=True)
        path = os.path.join(self.log_dir, "kpi_train.csv")
        with open(path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(self.rows[0].keys()))
            writer.writeheader()
            writer.writerows(self.rows)


def train_ppo(run_idx, seed=42):
    """Train a single PPO run with the given config index."""
    config = PPO_CONFIGS[run_idx]
    run_name = config["name"]
    print(f"\n{'='*60}\nTraining PPO - {run_name} (Run {run_idx})\n{'='*60}")

    log_dir = f"results/logs/{run_name}"
    model_dir = f"results/models/{run_name}"
    os.makedirs(log_dir, exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)

    env_kwargs = config.get("env_kwargs", {})
    train_env = Monitor(gym.make("AstroExploration-v0", **env_kwargs), log_dir)
    eval_env = Monitor(gym.make("AstroExploration-v0", **env_kwargs))

    model_params = {k: v for k, v in config.items() if k not in ["name", "env_kwargs"]}

    model = PPO(
        "MlpPolicy",
        train_env,
        verbose=1,
        seed=seed,
        tensorboard_log=f"results/logs/{run_name}_tb",
        **model_params,
    )

    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=model_dir,
        log_path=log_dir,
        eval_freq=EVAL_FREQ,
        n_eval_episodes=N_EVAL_EPISODES,
        deterministic=True,
    )
    kpi_callback = MissionKPITrainCallback(log_dir)

    start = time.time()
    model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=[eval_callback, kpi_callback], progress_bar=True)
    wall_time = time.time() - start

    model.save(os.path.join(model_dir, "final_model"))
    train_env.close()
    eval_env.close()

    print(f"PPO {run_name} done in {wall_time:.1f}s")
    return {"run_name": run_name, "wall_time": wall_time, "algorithm": "PPO"}


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--run", type=int, default=0)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    train_ppo(args.run, args.seed)


In [18]:
%%writefile training/train_reinforce.py
"""Train REINFORCE agent on AstroExploration environment."""

import os
import sys
import time
import argparse

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import astro_env  # noqa: F401
import gymnasium as gym
from agents.reinforce_agent import REINFORCEAgent
from training.hyperparams import REINFORCE_CONFIGS, REINFORCE_EPISODES


def train_reinforce(run_idx, seed=42):
    """Train a single REINFORCE run with the given config index."""
    config = REINFORCE_CONFIGS[run_idx]
    run_name = config["name"]
    print(f"\n{'='*60}\nTraining REINFORCE - {run_name} (Run {run_idx})\n{'='*60}")

    save_dir = f"results/models/{run_name}"
    os.makedirs(save_dir, exist_ok=True)

    env_kwargs = config.get("env_kwargs", {})
    env = gym.make("AstroExploration-v0", **env_kwargs)

    agent = REINFORCEAgent(
        env=env,
        learning_rate=config["learning_rate"],
        gamma=config["gamma"],
        hidden_sizes=config["hidden_sizes"],
        baseline=config.get("baseline", "mean"),
        entropy_coef=config.get("entropy_coef", 0.01),
        grad_clip_norm=config.get("grad_clip_norm", 1.0),
        seed=seed,
    )

    start = time.time()
    history, kpi = agent.train(
        num_episodes=REINFORCE_EPISODES,
        log_interval=100,
        save_dir=save_dir,
        curriculum_episodes=config.get("curriculum_episodes", 500),
    )
    wall_time = time.time() - start

    env.close()
    print(f"REINFORCE {run_name} done in {wall_time:.1f}s")

    # Lightweight KPI summary for quick comparison across runs.
    if len(kpi["episode_reward"]) >= 100:
        print(
            "KPI(last100) | "
            f"reward={sum(kpi['episode_reward'][-100:]) / 100:.2f}, "
            f"success={sum(kpi['success'][-100:]) / 100:.2f}, "
            f"detect={sum(kpi['detected_total'][-100:]) / 100:.2f}, "
            f"transmit={sum(kpi['transmitted_total'][-100:]) / 100:.2f}, "
            f"min_dist={sum(kpi['min_distance'][-100:]) / 100:.3f}, "
            f"compat={sum(kpi['compatible_rate'][-100:]) / 100:.2f}"
        )

    return {"run_name": run_name, "wall_time": wall_time, "algorithm": "REINFORCE"}


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--run", type=int, default=0)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    train_reinforce(args.run, args.seed)


## Verify Environment
Quick smoke-test to confirm the env loads and steps correctly before training.

In [19]:
# Propagate notebook FAST_MODE settings to hyperparams via env vars
import os
os.environ['TOTAL_TIMESTEPS']    = str(TOTAL_TIMESTEPS)
os.environ['REINFORCE_EPISODES'] = str(REINFORCE_EPISODES)

# Register and verify
import importlib, astro_env
importlib.reload(astro_env)
import gymnasium as gym

env = gym.make('AstroExploration-v0')
obs, info = env.reset(seed=0)
print('Observation shape :', obs.shape)
print('Action space      :', env.action_space)
print('Obs space         :', env.observation_space)

for _ in range(5):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)

env.close()
print('Environment OK — 5 random steps completed.')


## Train DQN - 10 tuned runs

DQN uses `ReducedDiscreteWrapper` to convert `MultiDiscrete([5,3,3,3,4,2])` -> `Discrete(360)`
(fixes roll to neutral, dropping 1 dimension). This gives **3x more samples per action** vs
the old `Discrete(1080)` wrapper at the same training budget.

Prioritized Experience Replay (PER) is enabled via `sb3-contrib.PrioritizedReplayBuffer` so
rare high-reward transitions (biosignature detection, transmission) are replayed more often.
Reward balance is rebalanced: collision penalty reduced to -80 so one collision no longer
overwhelms the +2000 success bonus.


In [ ]:
import sys
sys.path.insert(0, '/kaggle/working')

# All config (PER, reward kwargs, env kwargs) is now in training/hyperparams.py.
# Run cells 7, 12, 13, 14 first to materialise the latest source files.
from training.train_dqn import train_dqn
from training.hyperparams import DQN_CONFIGS

dqn_results = []
for run_idx in range(len(DQN_CONFIGS)):
    try:
        result = train_dqn(run_idx, seed=SEED)
        result['status'] = 'completed'
    except Exception as e:
        print(f'ERROR DQN run {run_idx}: {e}')
        result = {
            'run_name': DQN_CONFIGS[run_idx]['name'],
            'algorithm': 'DQN',
            'wall_time': 0,
            'status': 'failed',
            'error': str(e),
        }
    dqn_results.append(result)

print(f'DQN complete: {sum(r["status"] == "completed" for r in dqn_results)}/{len(dqn_results)} runs succeeded')


## Train PPO - 10 tuned runs

PPO handles `MultiDiscrete` natively (no wrapper).

In [ ]:
from training.train_ppo import train_ppo
from training.hyperparams import PPO_CONFIGS

ppo_results = []
for run_idx in range(len(PPO_CONFIGS)):
    try:
        result = train_ppo(run_idx, seed=SEED)
        result['status'] = 'completed'
    except Exception as e:
        print(f'ERROR PPO run {run_idx}: {e}')
        result = {'run_name': PPO_CONFIGS[run_idx]['name'], 'algorithm': 'PPO',
                  'wall_time': 0, 'status': 'failed', 'error': str(e)}
    ppo_results.append(result)

print(f'\nPPO complete: {sum(r["status"]=="completed" for r in ppo_results)}/'
      f'{len(ppo_results)} runs succeeded')


## Train REINFORCE - 10 tuned runs

Custom PyTorch implementation with entropy regularization, gradient clipping, and curriculum progression.

In [ ]:
from training.train_reinforce import train_reinforce
from training.hyperparams import REINFORCE_CONFIGS

reinforce_results = []
for run_idx in range(len(REINFORCE_CONFIGS)):
    try:
        result = train_reinforce(run_idx, seed=SEED)
        result['status'] = 'completed'
    except Exception as e:
        print(f'ERROR REINFORCE run {run_idx}: {e}')
        result = {'run_name': REINFORCE_CONFIGS[run_idx]['name'], 'algorithm': 'REINFORCE',
                  'wall_time': 0, 'status': 'failed', 'error': str(e)}
    reinforce_results.append(result)

print(f'\nREINFORCE complete: {sum(r["status"]=="completed" for r in reinforce_results)}/'
      f'{len(reinforce_results)} runs succeeded')


## Strategy Summary

Goal: shift mean rewards from negative to positive by improving mission completion, not just survival.

### Core levers already implemented
- Dense reward shaping toward nearest target.
- Instrument compatibility bonuses in detection zones.
- Reduced step penalties and increased transmission reward.
- Curriculum progression from navigation -> detection -> transmission.
- KPI logging for success rate, detections, transmissions, and minimum target distance.

### Expected trend
- Higher compatibility rate and lower minimum target distance first.
- Then higher detection/transmission counts.
- Positive mean reward follows once mission KPIs stabilize.

In [ ]:
# Implementation notes:
# The reward shaping, curriculum, mission KPI tracking, and ablation configurations
# are implemented directly in the writefile cells for:
# - astro_env/rewards.py
# - astro_env/astro_exploration_env.py
# - agents/reinforce_agent.py
# - training/hyperparams.py
# - training/train_dqn.py
# - training/train_ppo.py
# - training/train_reinforce.py
#
# Re-run those writefile cells, then run training cells for DQN/PPO/REINFORCE.

## Execution Checklist

1. Re-run all `%%writefile` cells to materialize latest code.
2. Run **Verify Environment** cell.
3. Run DQN/PPO/REINFORCE training cells.
4. Compare `kpi_train.csv` and `rewards.csv` across runs.
5. Keep configs that improve both mean reward and mission KPIs over multiple seeds.

## Save Experiment Index


In [ ]:
import json

all_results = dqn_results + ppo_results + reinforce_results
index = {
    'total_experiments': len(all_results),
    'total_wall_time_seconds': sum(r.get('wall_time', 0) for r in all_results),
    'seed': SEED,
    'fast_mode': FAST_MODE,
    'results': all_results,
}

index_path = 'results/experiment_index.json'
with open(index_path, 'w') as f:
    json.dump(index, f, indent=2)

print('Experiment index saved to', index_path)
print(f'Total wall time: {index["total_wall_time_seconds"]/3600:.2f} h')
completed = [r for r in all_results if r.get('status') == 'completed']
print(f'{len(completed)}/{len(all_results)} experiments completed successfully')

## Results Summary


In [ ]:
import glob
import os

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

# -- Wall-time summary ---------------------------------------------------------
rows = []
for r in all_results:
    rows.append({
        'Algorithm': r.get('algorithm', ''),
        'Run': r.get('run_name', ''),
        'Wall Time (s)': r.get('wall_time', 0),
        'Status': r.get('status', ''),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

summary = df[df['Status'] == 'completed'].groupby('Algorithm')['Wall Time (s)'].agg(['mean', 'sum'])
print('\nWall time summary (completed runs):')
print(summary)

# -- Learning curves -----------------------------------------------------------
def load_monitor(log_dir):
    for pattern in [f'{log_dir}/monitor.csv', f'{log_dir}/*.monitor.csv']:
        files = glob.glob(pattern)
        if not files:
            continue
        try:
            data = pd.read_csv(files[0], skiprows=1)
            if 'r' in data.columns:
                data = data.rename(columns={'r': 'reward', 'l': 'length', 't': 'time'})
            return data
        except Exception:
            pass
    return None


def load_reinforce_csv(model_dir):
    path = f'{model_dir}/rewards.csv'
    return pd.read_csv(path) if os.path.exists(path) else None


def smooth(values, window=50):
    adaptive = min(window, max(1, len(values) // 5))
    return pd.Series(values).rolling(window=adaptive, min_periods=1).mean().values


from training.hyperparams import DQN_CONFIGS, PPO_CONFIGS, REINFORCE_CONFIGS

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('AstroLogic - Learning Curves', fontsize=15, fontweight='bold')

algo_data = [
    ('DQN', DQN_CONFIGS, axes[0], False),
    ('PPO', PPO_CONFIGS, axes[1], False),
    ('REINFORCE', REINFORCE_CONFIGS, axes[2], True),
]

for algo_name, configs, ax, is_reinforce in algo_data:
    ax.set_title(algo_name, fontsize=13)
    ax.set_xlabel('Episode')
    ax.set_ylabel('Episode Reward')
    ax.grid(True, alpha=0.3)

    colors = plt.cm.tab10(range(len(configs)))
    for i, cfg in enumerate(configs):
        run_name = cfg['name']
        run_df = load_reinforce_csv(f'results/models/{run_name}') if is_reinforce else load_monitor(f'results/logs/{run_name}')
        if run_df is not None and 'reward' in run_df.columns:
            label = run_name.replace(f'{algo_name.lower()}_', '')
            ax.plot(smooth(run_df['reward'].values), color=colors[i], alpha=0.75, label=label, linewidth=0.9)

    ax.legend(fontsize=6, loc='lower right')

plt.tight_layout()
plt.savefig('results/plots/learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Learning curves saved.')

## Package Outputs

Zip everything in `results/` so it appears in the Kaggle output panel for download.

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/astro_results', 'zip', '/kaggle/working/results')
print('Archive created: /kaggle/working/astro_results.zip')
# List saved models
for root, dirs, files in os.walk('results/models'):
    for f in files:
        fpath = os.path.join(root, f)
        print(f'  {fpath}  ({os.path.getsize(fpath)//1024} KB)')
